<a href="https://colab.research.google.com/github/Y-o-o-Y/Factor-Backtest/blob/main/Factor_backtest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install giotto-tda plotly ecos

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.1/222.1 kB 5.6 MB/s eta 0:00:00


In [22]:
pip install PCA

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.4/190.4 kB 10.1 MB/s eta 0:00:00


In [34]:
# @title Factor test
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from google.colab import drive
import os
import ast
import statsmodels.api as sm
import cvxpy as cp
from sklearn.covariance import LedoitWolf
from scipy.optimize import minimize

# ==========================================
# 1. 數據處理模組
# ==========================================
class DataProcessor:
    @staticmethod
    def load_data(file_path):
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"❌ 找不到檔案：{file_path}")
        full_df = pd.read_parquet(file_path)
        df = full_df['Close'] if isinstance(full_df.columns, pd.MultiIndex) else full_df
        return df

    @staticmethod
    def load_industry_mapping(dict_path, columns):
        """讀取行業字典並自動對齊 Dataframe 的 Tickers """
        with open(dict_path, 'r') as f:
            content = f.read()
            # 找到 tickers = { ... } 的部分並解析
            dict_str = content.split('=', 1)[1].strip()
            raw_dict = ast.literal_eval(dict_str)

        # 翻轉字典：從 {行業: [Ticker]} 變成 {Ticker: 行業}
        ticker_to_ind = {}
        for industry, t_list in raw_dict.items():
            for t in t_list:
                ticker_to_ind[t] = industry

        # 自動對齊：將 columns (股票清單) 映射到行業，找不到的填 "Unknown"
        mapping_series = pd.Series(ticker_to_ind)
        aligned_mapping = mapping_series.reindex(columns).fillna("Unknown")
        return aligned_mapping

    @staticmethod
    def get_returns(df):
        # 統計用 log return，績效用 simple return [cite: 10, 13, 19]
        log_returns = np.log(df / df.shift(1))
        simple_returns = df.pct_change()
        return log_returns, simple_returns

class WQOperators:
    @staticmethod
    def ts_rank(df, window):
        """
        時間序列排名 (Normalized Rank)
        邏輯：計算當前值在過去 d 天的排名，並歸一化到 [0, 1]
        公式：(rank - 1) / (window - 1)
        """
        # 使用 rolling apply 獲取排名。pct=False 拿到原始排名 (1-based)
        # 注意：df.rank() 預設處理平手的方式是 'average'，符合邏輯
        def get_rank(x):
            return x.rank(pct=False).iloc[-1]

        rank = df.rolling(window).apply(get_rank)
        return (rank - 1) / (window - 1)

    @staticmethod
    def ts_decay_linear(df, window):
        """
        線性衰減移動平均 (Linearly Weighted Moving Average)
        邏輯：越近的數據權重越高，權重呈線性分佈 [1, 2, ..., d]
        """
        weights = np.arange(1, window + 1)
        sum_weights = weights.sum()

        def apply_decay(x):
            # x 是長度為 window 的數組，與 weights 進行點積
            return np.dot(x, weights) / sum_weights

        return df.rolling(window).apply(apply_decay, raw=True)


    @staticmethod
    def ts_sum(df, window):
        return df.rolling(window).sum()

    @staticmethod
    def ts_av_diff(df, window):
        """公式：x - ts_mean(x, d)"""
        return df - df.rolling(window).mean()

    @staticmethod
    def trade_when(trigger, value, exit_cond):
        """
        模擬 WQ trade_when(x, y, z)
        trigger (x): 觸發條件
        value (y): 新的 Alpha 值
        exit_cond (z): 退出條件
        """
        # 如果 exit_cond 是 0，代表不設退出條件
        if isinstance(exit_cond, (int, float)) and exit_cond == 0:
            exit_cond = pd.DataFrame(False, index=value.index, columns=value.columns)

        # 1. 建立一個容器，只有在觸發且未退出時填入新值 y
        alpha = pd.DataFrame(np.nan, index=value.index, columns=value.columns)
        alpha = alpha.where(~(trigger & ~exit_cond), value)

        # 2. 如果退出條件 z 成立，設為 NaN (平倉)
        alpha[exit_cond] = np.nan

        # 3. 使用 ffill 實現 "Hold previous value" 邏輯
        return alpha.ffill()

    @staticmethod
    def ts_delta(df, window):
        """時間序列差分：x - delay(x, d)"""
        return df.diff(window)

    @staticmethod
    def ts_corr(df1, df2, window):
        """時間序列相關性：兩個序列在 window 內的滾動相關係數"""
        # 使用 rolling corr 計算兩組 DataFrame 對應位置的相關性
        return df1.rolling(window).corr(df2)

# ==========================================
# 2. 回測引擎模組 (靈活中性化版本)
# ==========================================
class BacktestEngine:
    def __init__(self, factor_data, return_data, industry_mapping, capital=10000):
        self.factor_data = factor_data
        self.return_data = return_data
        self.industry_mapping = industry_mapping
        self.capital = capital

    def _apply_neutralization(self, date_slice, market_neutral, industry_neutral):
        """
        根據開關執行不同的中性化邏輯
        """
        # 如果都不開，直接回傳
        if not market_neutral and not industry_neutral:
            return date_slice

        # 準備基礎數據
        df = pd.DataFrame({
            'factor': date_slice,
            'industry': self.industry_mapping
        })

        # 樣本太少不處理
        if df['factor'].dropna().count() < 20:
            return date_slice

        # 情況 A: 同時開啟市場與行業中性 (使用 OLS 回歸取殘差)
        if market_neutral and industry_neutral:
            temp_df = df.dropna()
            X = pd.get_dummies(temp_df['industry'], drop_first=True, dtype=float)
            X = sm.add_constant(X)
            model = sm.OLS(temp_df['factor'], X).fit()
            residuals = model.resid
            return residuals.reindex(date_slice.index)

        # 情況 B: 僅市場中性 (橫截面去中心化)
        elif market_neutral:
            return date_slice - date_slice.mean()

        # 情況 C: 僅行業中性 (分組去中心化)
        elif industry_neutral:
            # 減去各行業自己的平均值
            return df.groupby('industry')['factor'].transform(lambda x: x - x.mean())

        return date_slice

    def run(self, start, end, market_neutral=True, industry_neutral=True):
        f_slice = self.factor_data.loc[start:end]
        r_slice = self.return_data.loc[start:end]

        # 執行中性化處理
        status_msg = f"Market={'ON' if market_neutral else 'OFF'}, Industry={'ON' if industry_neutral else 'OFF'}"
        print(f"💡 正在執行回測 | 中性化: {status_msg}")

        # 使用 apply 處理每一天的因子
        f_processed = f_slice.apply(
            self._apply_neutralization,
            axis=1,
            args=(market_neutral, industry_neutral)
        )

        # 2. Rank 並達成 Dollar Neutral
        # 注意：如果已經做過中性化，rank - 0.5 會更乾淨
        weights = f_processed.rank(axis=1, pct=True) - 0.5

        # 3. 歸一化
        weights = weights.div(weights.abs().sum(axis=1), axis=0).fillna(0)

        # 策略損益計算 (使用 shift(1) 避免 look-ahead bias)
        strat_ret = (weights.shift(1) * r_slice).sum(axis=1)
        cum_pnl = (1 + strat_ret).cumprod() * self.capital

        metrics = self._calculate_metrics(strat_ret, cum_pnl, weights)
        return {"pnl": cum_pnl, "metrics": metrics, "weights": weights, "strat_ret": strat_ret}

    def _calculate_metrics(self, strat_ret, cum_pnl, weights):
        # ... (保持原本的計算 logic)
        total_ret = (cum_pnl.iloc[-1] / self.capital) - 1
        ann_ret = (1 + total_ret) ** (252 / len(strat_ret)) - 1
        sharpe = np.sqrt(252) * strat_ret.mean() / (strat_ret.std() + 1e-9)
        dd = (cum_pnl / cum_pnl.cummax()) - 1
        max_dd = dd.min()
        weight_diff = weights.diff().abs().sum(axis=1)
        daily_turnover = weight_diff.mean() / (weights.abs().sum(axis=1).mean() + 1e-9)
        total_traded = (weight_diff * self.capital).sum()
        margin_bp = (strat_ret.sum() * self.capital / total_traded * 10000) if total_traded != 0 else 0
        return [total_ret, sharpe, ann_ret, daily_turnover, max_dd, margin_bp]

class PortfolioOptimizer:
    def __init__(self, lambda_reg=0.5, max_weight=0.05, leverage=1.0):
        self.lambda_reg = lambda_reg
        self.max_weight = max_weight
        self.leverage = leverage
        self.lw = LedoitWolf()

    def estimate_risk(self, history_returns):
        # 獨立出風險矩陣計算
        return self.lw.fit(history_returns).covariance_

    def optimize(self, alpha_vector, sigma):
        # 直接接收 sigma，不再重複計算
        tickers = alpha_vector.index
        n = len(tickers)
        w = cp.Variable(n)

        risk = cp.quad_form(w, sigma)
        objective = cp.Maximize(alpha_vector.values @ w - self.lambda_reg * risk)

        constraints = [
            cp.sum(w) == 0,
            cp.norm(w, 1) <= self.leverage,
            cp.abs(w) <= self.max_weight
        ]

        prob = cp.Problem(objective, constraints)

        try:
            prob.solve(solver=cp.OSQP)
            if w.value is None:
                prob.solve(solver=cp.SCS)
        except Exception:
            try:
                prob.solve()
            except:
                return pd.Series(0, index=tickers)

        if w.value is None:
            return pd.Series(0, index=tickers)

        return pd.Series(w.value, index=tickers)



# ==========================================
# 3. 視覺化模組
# ==========================================
class Visualizer:
    @staticmethod
    def generate_yearly_report(res, asset_returns):
        """
        生成 WQ 風格的逐年觀察報告 (含 IC, Rank IC, ICIR)
        asset_returns: 原始股票收益率矩陣 (用於計算 IC)
        """
        weights = res['weights']
        strat_returns = res['strat_ret']
        years = sorted(weights.index.year.unique())
        report_data = []

        next_ret = asset_returns.shift(-1)

        for year in years:
            # 取得該年度區間數據
            y_w = weights[weights.index.year == year]
            y_r = strat_returns[strat_returns.index.year == year]
            y_asset_ret = next_ret[next_ret.index.year == year]

            if len(y_r) == 0: continue

            # --- 基礎績效指標 ---
            ann_ret = y_r.mean() * 252
            sharpe = (y_r.mean() / (y_r.std() + 1e-9)) * np.sqrt(252)

            y_diff = y_w.diff().abs().sum(axis=1)
            y_gross = y_w.abs().sum(axis=1).replace(0, 1)
            daily_turnover = (y_diff / y_gross).mean()

            # --- IC 相關計算 (橫截面) ---
            # Normal IC (Pearson)
            daily_ic = y_w.corrwith(y_asset_ret, axis=1, method='pearson')
            # Rank IC (Spearman)
            daily_rank_ic = y_w.corrwith(y_asset_ret, axis=1, method='spearman')

            ic_mean = daily_ic.mean()
            rank_ic_mean = daily_rank_ic.mean()
            icir = (ic_mean / (daily_ic.std() + 1e-9)) * np.sqrt(252) # ICIR = Mean / Std

            # WQ Fitness 公式
            fitness_denom = max(daily_turnover, 0.125)
            fitness = sharpe * np.sqrt(abs(ann_ret) / fitness_denom)

            cum_pnl = (1 + y_r).cumprod()
            max_dd = ((cum_pnl / cum_pnl.cummax()) - 1).min()
            margin_bp = (y_r.sum() / (y_diff.sum() + 1e-9) * 10000)

            report_data.append({
                "Year": year,
                "Rank IC": round(rank_ic_mean, 4),
                "IR": round(icir, 2),
                "Sharpe": round(sharpe, 2),
                "Fitness": round(fitness, 2),
                "Returns": f"{ann_ret:.2%}",
                "Turnover": f"{daily_turnover:.2%}",
                "Drawdown": f"{abs(max_dd):.2%}",
                "Margin": f"{margin_bp:.2f} bp"
            })

        return pd.DataFrame(report_data).set_index("Year")

    @staticmethod
    def plot_pnl(is_res, oos_res):
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=is_res['pnl'].index, y=is_res['pnl'], name='IS', line=dict(color='cyan')))
        fig.add_trace(go.Scatter(x=oos_res['pnl'].index, y=oos_res['pnl'], name='OOS', line=dict(color='orange')))
        fig.update_layout(title='IS vs OOS PnL', template='plotly_dark')
        fig.show()

    @staticmethod
    def plot_hl_ladder(factor, returns, n_bins=5):
        # 1. 確保因子與回報數據對齊，避免計算偏離
        common_idx = factor.index.intersection(returns.index)
        factor = factor.loc[common_idx]
        returns = returns.loc[common_idx]

        # 2. 計算 Rank
        ranks = factor.rank(axis=1, pct=True)
        fig = go.Figure()

        for i in range(n_bins):
            # 構建該 Bin 的 Mask
            lower = i / n_bins
            upper = (i + 1) / n_bins
            mask = (ranks > lower) & (ranks <= upper)

            # 【核心修復】：使用 .where() 排除不屬於該 Bin 的股票，
            # 這樣 mean(axis=1) 才會只計算該組內的股票平均值
            # 並且使用 shift(1) 避免未來函數
            bin_ret = returns.where(mask.shift(1)).mean(axis=1)

            # 處理 NaN，確保累計收益不會斷掉
            cum_ret = (1 + bin_ret.fillna(0)).cumprod()

            fig.add_trace(go.Scatter(
                x=cum_ret.index,
                y=cum_ret,
                name=f'Q{i+1} (High)' if i == n_bins-1 else f'Q{i+1}'
            ))

        fig.update_layout(
            title='H-L Ladder (In Sample)',
            xaxis_title='Date',
            yaxis_title='Cumulative Return',
            template='plotly_dark'
        )
        fig.show()

# ==========================================
# 4. 主程式控制
# ==========================================
if __name__ == "__main__":
    drive.mount('/content/drive')
    BASE_DIR = "/content/drive/MyDrive/Colab Notebooks/WQ factor"
    PRICE_PATH = os.path.join(BASE_DIR, "1100_stocks_cleaned.parquet")
    DICT_PATH = os.path.join(BASE_DIR, "ticker_dictionary_cleaned.txt")

    # --- 配置區：在這裡一鍵切換模式 ---
    MKT_NEU = True   # 是否開啟市場中性
    IND_NEU = False   # 是否開啟行業中性
    IS_START, IS_END = "2021-01-01", "2023-12-31"
    OOS_START, OOS_END = "2024-01-01", "2026-12-31"
    # ------------------------------

    # 1. 讀取與對齊
    dp = DataProcessor()
    close_df = dp.load_data(PRICE_PATH)
    industry_mapping = dp.load_industry_mapping(DICT_PATH, close_df.columns)
    log_ret, simple_ret = dp.get_returns(close_df)

    ## 2. 因子計算

    ## vol_AR(1) 1
    # vol_sq = log_ret**2
    # factor = -vol_sq.rolling(window=252).corr(vol_sq.shift(1))
    # ==========================================

    ## ==========================================
    # STO 2
    # ==========================================
    # 獲取最高、最低、收盤價
    full_df = pd.read_parquet(PRICE_PATH)
    high_df = full_df['High']
    low_df = full_df['Low']
    close_df = full_df['Close']

    ## 運算子初始化
    wq = WQOperators()

    # 1. 計算 STO
    # STO = 100 * (close - ts_min(low, n)) / (ts_max(high, n) - ts_min(low, n))
    lookback = 1 #當天
    lowest_low = low_df.rolling(lookback).min()
    highest_high = high_df.rolling(lookback).max()
    sto = 100 * (close_df - lowest_low) / (highest_high - lowest_low + 1e-9)

    # 2. 計算 alpha = if_else(STO < 20, 1, if_else(STO > 80, -1, 0))
    alpha = pd.DataFrame(0, index=sto.index, columns=sto.columns)
    alpha[sto < 20] = 1
    alpha[sto > 80] = -1

    # 3. 嵌套運算：ts_decay_linear(ts_rank(alpha, 60), 20)
    ranked_alpha = wq.ts_rank(alpha, 60)
    factor = wq.ts_decay_linear(ranked_alpha, 20)
    ## ==========================================

    ## CMF 3
    # full_df = pd.read_parquet(PRICE_PATH)
    # close = full_df['Close']
    # high = full_df['High']
    # low = full_df['Low']
    # volume = full_df['Volume']

    # wq = WQOperators()

    # # 1. 計算 CMF (Chaikin Money Flow)
    # # MFM = ((close-low)-(high-low))/(high-low)
    # # 注意：(close-low)-(high-low) 簡化後等於 (close-high)
    # mfm = (close - high) / (high - low + 1e-9)
    # mfv = mfm * volume
    # cmf = wq.ts_sum(mfv, 20) / (wq.ts_sum(volume, 20) + 1e-9)

    # # 2. 計算內部元件：rank(ts_av_diff(-CMF, 252))
    # # 這裡的 rank 是橫截面 rank (cross-sectional)
    # inner_val = wq.ts_av_diff(-cmf, 252)
    # y_value = inner_val.rank(axis=1, pct=True)

    # # 3. 計算 ADV20
    # adv20 = volume.rolling(20).mean()

    # # 4. 執行 trade_when(volume > adv20, y_value, 0)
    # # 邏輯：當今日成交量爆量 ( > adv20) 時，更新持倉權重，否則維持昨日權重
    # factor = wq.trade_when(volume > adv20, y_value, 0)
    ## ==========================================

    ## Corr(delta_abs_return, return) 4
    # wq = WQOperators()

    # # 1. 準備組件
    # # abs(returns) -> 使用簡單回報的絕對值
    # abs_ret = simple_ret.abs()

    # # ts_delta(abs(returns), 1) -> 波動幅度的變化
    # vola_delta = wq.ts_delta(abs_ret, 1)

    # # log(1 + returns) -> 即 log_ret
    # # 這裡直接使用之前算的 log_ret [cite: 2026-01-09, 2026-03-20]

    # # 2. 計算相關性 ts_corr(..., 60)
    # correlation = wq.ts_corr(vola_delta, log_ret, 60)

    # # 3. 最終因子：-rank(...)
    # # 注意：這裡的 rank 是橫截面 rank
    # factor = -correlation.rank(axis=1, pct=True)


    # ==========================================
    # Return AR(2) Correlation 5
    # target_ret = log_ret

    # # 2. 計算二階延遲項 (ts_delay = 2)
    # delay_ret_2 = target_ret.shift(2)

    # # 3. 計算 252 日（約一年）滾動相關性 ts_corr
    # # 我們直接調用 wq 算子或使用 pandas 滾動計算
    # ar2_corr = target_ret.rolling(window=252).corr(delay_ret_2)

    # # 4. 執行橫截面 Rank 並取負號
    # # 根據 WQ 慣例，負號通常代表捕捉「均值回歸」或是對沖某種過度反應
    # factor = -ar2_corr.rank(axis=1, pct=True)
    # # 5. 處理結果並放入回測引擎
    # factor = factor.fillna(0)

    # ==========================================
		# Vol_Ratio (Vol 10 / Vol 60) 6
		# ==========================================
		# 1. 計算 10 日與 60 日歷史波動率 (使用 stdev 作為代理)
    # vol10 = log_ret.rolling(window=10).std()
    # vol60 = log_ret.rolling(window=60).std()

		# # 2. 計算波動率比率並執行橫截面 Rank
		# # 依據你的指令，直接忽略 ts_backfill
    # vol_ratio = vol10 / vol60
    # factor = vol_ratio.rank(axis=1, pct=True)
		# ==========================================


    # ==========================================
		# 因子：-Skewness 7
		# ==========================================

		# 1. 計算分子：20 日平均的三次偏離矩 (3rd Central Moment)
		# 使用你定義的 log_ret 作為基礎 [cite: 2026-03-20]
    # wq = WQOperators()
    # rolling_mean20 = log_ret.rolling(window=20).mean()
    # skew_num = ((log_ret - rolling_mean20) ** 3).rolling(window=20).mean()

		# # 2. 計算分母：100 日滾動標準差的立方 (加上 epsilon 防止除以零)
    # rolling_std100 = log_ret.rolling(window=100).std()
    # skew_den = (rolling_std100 + 0.000001) ** 3

		# # 3. 計算原始偏度值 (Raw Skewness)
    # skew_raw = skew_num / skew_den

		# # 4. 執行 20 日時間序列排名 (ts_rank) 並取負號
		# # 調用你 WQOperators 中的 ts_rank 算子
    # factor = -wq.ts_rank(skew_raw, 20)

    # 3. 初始化引擎
    bt = BacktestEngine(factor, simple_ret, industry_mapping)

    # 執行樣本內 (IS)
    print(f"\n🚀 執行樣本內回測 ({IS_START} ~ {IS_END})...")
    is_res = bt.run(IS_START, IS_END, market_neutral=MKT_NEU, industry_neutral=IND_NEU)

    # 執行樣本外 (OOS)
    print(f"\n🚀 執行樣本外回測 ({OOS_START} ~ {OOS_END})...")
    oos_res = bt.run(OOS_START, OOS_END, market_neutral=MKT_NEU, industry_neutral=IND_NEU)

    # 4. 輸出與視覺化
    viz = Visualizer()

    # 顯示年度報告 (樣本內)
    print("\n[ In-Sample Yearly Report ]")
    display(viz.generate_yearly_report(is_res, log_ret.loc[IS_START:IS_END]))
    print("\n[ Out-Sample Yearly Report ]")
    display(viz.generate_yearly_report(oos_res, log_ret.loc[OOS_START:OOS_END]))

    # 畫 PnL 曲線
    viz.plot_pnl(is_res, oos_res)

    # 畫 H-L Ladder (注意：為了讓圖表反應中性化的威力，建議傳入 is_res 算好的權重排名)
    # 這裡直接用 is_res['weights'] 來反映你真正交易的「階梯」
    print("\n📊 正在生成 H-L Ladder...")
    viz.plot_hl_ladder(is_res['weights'], simple_ret.loc[IS_START:IS_END])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

🚀 執行樣本內回測 (2021-01-01 ~ 2023-12-31)...
💡 正在執行回測 | 中性化: Market=ON, Industry=OFF

🚀 執行樣本外回測 (2024-01-01 ~ 2026-12-31)...
💡 正在執行回測 | 中性化: Market=ON, Industry=OFF

[ In-Sample Yearly Report ]


,Rank IC,IR,Sharpe,Fitness,Returns,Turnover,Drawdown,Margin
Year,,,,,,,,
2021,0.0074,1.12,1.12,0.42,4.87%,33.87%,2.99%,5.71 bp
2022,0.0091,1.39,1.35,0.62,7.06%,33.81%,4.10%,8.29 bp
2023,0.0066,1.37,0.95,0.27,2.87%,34.76%,1.43%,3.28 bp



[ Out-Sample Yearly Report ]


,Rank IC,IR,Sharpe,Fitness,Returns,Turnover,Drawdown,Margin
Year,,,,,,,,
2024,0.0052,0.68,0.14,0.02,0.45%,33.39%,1.94%,0.53 bp
2025,0.0056,0.75,0.60,0.15,2.19%,34.50%,2.85%,2.52 bp
2026,0.0144,2.27,2.20,1.05,7.93%,34.46%,1.31%,9.13 bp



📊 正在生成 H-L Ladder...


In [3]:
# @title 因子信號打包
# ==========================================
# 5. 因子回報提取模組 (為 MW 算法準備數據)
# ==========================================

# 1. 初始化容器
all_factor_signals = {}
wq = WQOperators()

print("🔍 開始計算 7 個因子的原始信號...")

# --- Factor 1: Vol_AR(1) ---
vol_sq = log_ret**2
all_factor_signals['Vol_AR1'] = -vol_sq.rolling(window=252).corr(vol_sq.shift(1))

# --- Factor 2: STO ---
high_df, low_df = close_df, close_df # 假設只有 Close，依據 WQ 慣例可代理
# 若有 High/Low 則建議由 DataProcessor 讀取，此處暫用 Close 示意邏輯
lowest_low = low_df.rolling(1).min()
highest_high = high_df.rolling(1).max()
sto = 100 * (close_df - lowest_low) / (highest_high - lowest_low + 1e-9)
alpha_sto = pd.DataFrame(0, index=sto.index, columns=sto.columns)
alpha_sto[sto < 20] = 1
alpha_sto[sto > 80] = -1
all_factor_signals['STO'] = wq.ts_decay_linear(wq.ts_rank(alpha_sto, 60), 20)

# --- Factor 3: CMF (簡化邏輯提取) ---
# 注意：這裡假設 volume 已讀取
if 'Volume' in locals() or 'volume' in locals():
    v = volume if 'volume' in locals() else Volume
    mfm = (close_df - close_df.rolling(1).max()) / (close_df.rolling(1).max() - close_df.rolling(1).min() + 1e-9)
    mfv = mfm * v
    cmf = wq.ts_sum(mfv, 20) / (wq.ts_sum(v, 20) + 1e-9)
    inner_val = wq.ts_av_diff(-cmf, 252)
    y_val = inner_val.rank(axis=1, pct=True)
    all_factor_signals['CMF'] = wq.trade_when(v > v.rolling(20).mean(), y_val, 0)
else:
    all_factor_signals['CMF'] = pd.DataFrame(0, index=close_df.index, columns=close_df.columns)

# --- Factor 4: Corr(delta_abs_return, return) ---
abs_ret = simple_ret.abs()
vola_delta = wq.ts_delta(abs_ret, 1)
correlation = wq.ts_corr(vola_delta, log_ret, 60)
all_factor_signals['Corr_Delta'] = -correlation.rank(axis=1, pct=True)

# --- Factor 5: Return AR(2) ---
ar2_corr = log_ret.rolling(window=252).corr(log_ret.shift(2))
all_factor_signals['Return_AR2'] = -ar2_corr.rank(axis=1, pct=True)

# --- Factor 6: Vol_Ratio ---
vol10 = log_ret.rolling(window=10).std()
vol60 = log_ret.rolling(window=60).std()
all_factor_signals['Vol_Ratio'] = (vol10 / vol60).rank(axis=1, pct=True)

# --- Factor 7: -Skewness ---
rolling_mean20 = log_ret.rolling(window=20).mean()
skew_num = ((log_ret - rolling_mean20) ** 3).rolling(window=20).mean()
rolling_std100 = log_ret.rolling(window=100).std()
skew_raw = skew_num / ((rolling_std100 + 1e-9) ** 3)
all_factor_signals['Skewness'] = -wq.ts_rank(skew_raw, 20)

print("✅ 因子信號計算完成。")

# 2. 透過回測引擎提取每個因子的每日收益率 (R_i)
# 我們統一使用 IS 區間進行實驗準備
factor_returns_dict = {}
bt_for_extraction = BacktestEngine(None, simple_ret, industry_mapping)

print(f"📊 正在提取各因子在 {IS_START} 至 {OOS_END} 的每日收益率...")

for name, signal in all_factor_signals.items():
    bt_for_extraction.factor_data = signal.fillna(0)
    # 執行回測以獲得 strat_ret [cite: 148, 150]
    res = bt_for_extraction.run(IS_START, OOS_END, market_neutral=MKT_NEU, industry_neutral=IND_NEU)
    factor_returns_dict[name] = res['strat_ret']

# 3. 合併為 Factor Returns DataFrame (即 MW 的 Oracle 輸入數據)
factor_returns_df = pd.DataFrame(factor_returns_dict).fillna(0)

# 4. 顯示各因子基礎統計
print("\n[ 各因子的獨立表現總結 ]")
summary = []
for col in factor_returns_df.columns:
    r = factor_returns_df[col]
    sharpe = np.sqrt(252) * r.mean() / (r.std() + 1e-9)
    ann_ret = r.mean() * 252
    summary.append({"Factor": col, "Ann_Ret": f"{ann_ret:.2%}", "Sharpe": round(sharpe, 2)})

display(pd.DataFrame(summary))

🔍 開始計算 7 個因子的原始信號...
✅ 因子信號計算完成。
📊 正在提取各因子在 2021-01-01 至 2026-12-31 的每日收益率...
💡 正在執行回測 | 中性化: Market=ON, Industry=OFF
💡 正在執行回測 | 中性化: Market=ON, Industry=OFF
💡 正在執行回測 | 中性化: Market=ON, Industry=OFF
💡 正在執行回測 | 中性化: Market=ON, Industry=OFF
💡 正在執行回測 | 中性化: Market=ON, Industry=OFF
💡 正在執行回測 | 中性化: Market=ON, Industry=OFF
💡 正在執行回測 | 中性化: Market=ON, Industry=OFF

[ 各因子的獨立表現總結 ]


,Factor,Ann_Ret,Sharpe
0,Vol_AR1,-0.29%,-0.10
1,STO,18.21%,1.13
2,CMF,18.21%,1.13
3,Corr_Delta,0.47%,0.12
4,Return_AR2,-0.16%,-0.06
5,Vol_Ratio,1.22%,0.38
6,Skewness,2.12%,0.54


In [15]:
# @title 等權重合成 (Baseline)
# ==========================================
# 6. 等權重因子合成 (Equal Weight Ensemble) - Baseline
# ==========================================

# 1. 取得所有因子名稱
factor_names = list(all_factor_signals.keys())
print(f"🧩 正在執行等權重合成，包含因子: {factor_names}")

# 2. 合成信號：將所有因子的 Rank 信號進行簡單平均
# 這裡使用 sum / count 的邏輯，對齊所有股票與日期
ew_signal = pd.concat(all_factor_signals.values(), axis=0, keys=factor_names)
ew_combined = ew_signal.groupby(level=1).mean()

# 3. 初始化回測引擎
bt_ew = BacktestEngine(ew_combined, simple_ret, industry_mapping)

# 4. 執行樣本內 (IS) 回測
print(f"\n🚀 [EW Baseline] 執行樣本內回測 ({IS_START} ~ {IS_END})...")
ew_is_res = bt_ew.run(IS_START, IS_END, market_neutral=MKT_NEU, industry_neutral=IND_NEU)

# 5. 執行樣本外 (OOS) 回測
print(f"\n🚀 [EW Baseline] 執行樣本外回測 ({OOS_START} ~ {OOS_END})...")
ew_oos_res = bt_ew.run(OOS_START, OOS_END, market_neutral=MKT_NEU, industry_neutral=IND_NEU)

# 6. 輸出報告與視覺化
viz = Visualizer()

print("\n[ Equal Weight - IS 年度報告 ]")
display(viz.generate_yearly_report(ew_is_res, log_ret.loc[IS_START:IS_END]))

print("\n[ Equal Weight - OOS 年度報告 ]")
display(viz.generate_yearly_report(ew_oos_res, log_ret.loc[OOS_START:OOS_END]))

# 畫出 PnL 曲線進行對比
viz.plot_pnl(ew_is_res, ew_oos_res)

# 觀察合成後的 H-L Ladder
print("\n📊 正在生成等權重合成因子的 H-L Ladder (IS)...")
viz.plot_hl_ladder(ew_is_res['weights'], simple_ret.loc[IS_START:IS_END])

🧩 正在執行等權重合成，包含因子: ['Vol_AR1', 'STO', 'CMF', 'Corr_Delta', 'Return_AR2', 'Vol_Ratio', 'Skewness']

🚀 [EW Baseline] 執行樣本內回測 (2021-01-01 ~ 2023-12-31)...
💡 正在執行回測 | 中性化: Market=ON, Industry=OFF

🚀 [EW Baseline] 執行樣本外回測 (2024-01-01 ~ 2026-12-31)...
💡 正在執行回測 | 中性化: Market=ON, Industry=OFF

[ Equal Weight - IS 年度報告 ]


,Rank IC,IR,Sharpe,Fitness,Returns,Turnover,Drawdown,Margin
Year,,,,,,,,
2021,0.0069,1.16,0.84,0.26,2.92%,30.36%,2.49%,3.82 bp
2022,0.0013,0.63,1.17,0.48,4.90%,29.91%,2.75%,6.50 bp
2023,0.0068,0.87,0.75,0.20,2.07%,28.87%,2.49%,2.85 bp



[ Equal Weight - OOS 年度報告 ]


,Rank IC,IR,Sharpe,Fitness,Returns,Turnover,Drawdown,Margin
Year,,,,,,,,
2024,0.0039,0.22,0.01,0.00,0.02%,28.46%,2.55%,0.03 bp
2025,-0.0027,-0.34,-0.60,-0.17,-2.24%,27.91%,4.20%,-3.18 bp
2026,0.0090,0.78,0.87,0.31,3.53%,28.17%,1.62%,4.97 bp



📊 正在生成等權重合成因子的 H-L Ladder (IS)...


In [26]:
#@title MW 動態合成
import numpy as np
import pandas as pd

# ==========================================
# 7. MW 權重優化器模組 (Multiplicative Weights Optimizer)
# ==========================================
class MWOptimizer:
    def __init__(self, n_experts, eta=0.01, rho=0.05, mode='linear'):
        """
        n_experts: 因子數量
        eta: 學習率 (learning rate), 論文建議 eta <= 1/2
        rho: 寬度參數 (width), 用於將回報縮放到 [-1, 1] 區間
        mode: 'linear' (標準 MW) 或 'hedge' (指數更新)
        """
        self.n = n_experts
        self.eta = eta
        self.rho = rho
        self.mode = mode
        # 初始化權重為 1
        self.weights = np.ones(n_experts)
        self.weight_history = []

    def get_distribution(self):
        """計算當前的概率分佈 (歸一化權重) """
        phi = np.sum(self.weights) # 勢函數 (Potential Function) [cite: 167]
        return self.weights / phi

    def update(self, returns):
        """
        根據當日收益率更新權重
        returns: 各因子的當日多空收益率向量
        """
        # 將收益率轉化為成本 (Cost) m_i = -R_i / rho [cite: 415]
        # 確保成本落在 [-1, 1] 之間 [cite: 145, 155]
        costs = -np.array(returns) / self.rho
        costs = np.clip(costs, -1, 1) # 強制符合邊界條件

        if self.mode == 'linear':
            # 標準線性更新: w = w * (1 - eta * cost)
            self.weights = self.weights * (1 - self.eta * costs)
        elif self.mode == 'hedge':
            # Hedge 指數更新: w = w * exp(-eta * cost) [cite: 202]
            self.weights = self.weights * np.exp(-self.eta * costs)

        # 紀錄權重歷史以便觀察收斂
        self.weight_history.append(self.weights.copy())

    def optimize(self, factor_returns_df):
        """
        對整個收益序列執行在線優化
        """
        dynamic_weights = []

        for date, row in factor_returns_df.iterrows():
            # 1. 紀錄今日開盤前的權重分佈
            current_p = self.get_distribution()
            dynamic_weights.append(current_p)

            # 2. 觀察今日收盤表現並更新權重 (用於明天)
            self.update(row.values)

        weight_df = pd.DataFrame(dynamic_weights, index=factor_returns_df.index, columns=factor_returns_df.columns)
        return weight_df

# ==========================================
# 8. 執行面板與參數控制 (Main Control Panel)
# ==========================================
if __name__ == "__main__":
    # --- [參數面板] ---
    # 寬度 rho: 建議設為因子歷史最大單日回報 (例如 0.05 代表 5%) [cite: 145, 155]
    RHO = 0.05

    # 學習率 eta: 論文建議依據 T 調整，或設為固定小值 [cite: 876]
    # 這裡我們先手動設定，稍後可以試試 sqrt(ln(n)/T)
    ETA = 0.1

    MODE = 'hedge' # 'linear' 或 'hedge(指數)'
    # -----------------

    print(f"⚙️ 啟動 MW 優化器 | 參數: Eta={ETA}, Rho={RHO}, Mode={MODE}")

    # 1. 執行 MW 在線優化 (利用已有的 factor_returns_df)
mwo = MWOptimizer(n_experts=len(factor_names), eta=ETA, rho=RHO, mode=MODE)
# 這裡直接用你上一個單元格算好的數據
mw_weights_path = mwo.optimize(factor_returns_df)

# 2. 修正後的信號合成 (核心修復：使用 .mul(..., axis=0) 確保日期對齊)
# 我們直接拿內存中的 all_factor_signals 來加權
combined_signal_mw = pd.DataFrame(0, index=close_df.index, columns=close_df.columns)

for name in factor_names:
    # mw_weights_path[name] 是 (日期,) 的 Series
    # all_factor_signals[name] 是 (日期, 股票) 的 DataFrame
    # 使用 mul(axis=0) 強制將權重對齊到每一天
    weighted_sig = all_factor_signals[name].mul(mw_weights_path[name], axis=0)
    combined_signal_mw = combined_signal_mw.add(weighted_sig.fillna(0))

# 3. 運行回測
print(f"🚀 In Sample")
mw_res = BacktestEngine(combined_signal_mw, simple_ret, industry_mapping).run(
    IS_START,
    IS_END,
    market_neutral=MKT_NEU,
    industry_neutral=IND_NEU
)

print(f"🚀 Out of Sample")
mw_res_oos = BacktestEngine(combined_signal_mw, simple_ret, industry_mapping).run(
    OOS_START,
    OOS_END,
    market_neutral=MKT_NEU,
    industry_neutral=IND_NEU
)

# 4. 輸出報告
print("\n[ MW - IS 年度報告 ]")
display(viz.generate_yearly_report(mw_res, log_ret.loc[IS_START:IS_END]))

print("\n[ MW - OOS 年度報告 ]")
display(viz.generate_yearly_report(mw_res_oos, log_ret.loc[OOS_START:OOS_END]))

print("\n📈 正在生成 PnL 績效對比圖...")
viz.plot_pnl(mw_res, mw_res_oos)

# 5. 可視化權重演變
fig_w = go.Figure()
for col in mw_weights_path.columns:
    fig_w.add_trace(go.Scatter(x=mw_weights_path.index, y=mw_weights_path[col], name=col, stackgroup='one'))
fig_w.update_layout(title="MW Dynamic Weight Allocation", yaxis_title="Weight %", template='plotly_dark')
fig_w.show()

⚙️ 啟動 MW 優化器 | 參數: Eta=0.1, Rho=0.05, Mode=hedge
🚀 In Sample
💡 正在執行回測 | 中性化: Market=ON, Industry=OFF
🚀 Out of Sample
💡 正在執行回測 | 中性化: Market=ON, Industry=OFF

[ MW - IS 年度報告 ]


,Rank IC,IR,Sharpe,Fitness,Returns,Turnover,Drawdown,Margin
Year,,,,,,,,
2021,0.0064,1.12,0.82,0.25,2.83%,29.94%,2.51%,3.75 bp
2022,0.0017,0.64,1.15,0.46,4.77%,29.68%,2.78%,6.38 bp
2023,0.0064,0.79,0.67,0.17,1.83%,28.35%,2.45%,2.56 bp



[ MW - OOS 年度報告 ]


,Rank IC,IR,Sharpe,Fitness,Returns,Turnover,Drawdown,Margin
Year,,,,,,,,
2024,0.0038,0.18,-0.04,-0.00,-0.11%,28.39%,2.60%,-0.15 bp
2025,-0.0024,-0.31,-0.56,-0.15,-2.06%,28.95%,4.01%,-2.82 bp
2026,0.0093,0.81,0.83,0.28,3.42%,30.04%,1.60%,4.52 bp



📈 正在生成 PnL 績效對比圖...


In [19]:
# @title Portfolio optimize

optimizer = PortfolioOptimizer(lambda_reg=0.5, max_weight=0.05)
common_dates = log_ret.index.intersection(all_factor_signals['Vol_AR1'].index)
test_dates = common_dates[126:]

# --- 核心修改：篩選每月最後一個交易日 ---
# 將 DatetimeIndex 轉為 Series 後按年月分組，取每組最後一個元素
rebalance_dates = test_dates.to_series().groupby([test_dates.year, test_dates.month]).tail(1).index

weights_vA_list = {}
weights_vB_list = {}

print(f" 🚀  優化器啟動 (每月調倉模式) | 處理月份數: {len(rebalance_dates)} | 域名檢查中...")

for d in rebalance_dates:
    try:
        # 1. 截面數據提取與對齊
        cs_factors = pd.DataFrame({
            name: df.loc[d] for name, df in all_factor_signals.items()
        }).dropna(how='all')

        available_tickers = cs_factors.index.intersection(log_ret.columns)
        cs_factors = cs_factors.loc[available_tickers]

        # Z-score 標準化 [cite: 698]
        cs_zscore = cs_factors.apply(lambda x: (x - x.mean()) / (x.std() + 1e-9), axis=0).fillna(0)

        # 2. 因子合成 (等權 vA 與 MW vB) [cite: 701, 705]
        alpha_vA = cs_zscore.mean(axis=1)

        if 'mw_weights_path' in locals():
            w_mw = mw_weights_path.loc[d] if d in mw_weights_path.index else mw_weights_path.iloc[-1]
            alpha_vB = (cs_zscore * w_mw).sum(axis=1)
        else:
            alpha_vB = alpha_vA

        # 3. 風險矩陣計算 (使用當天往回推 126 天的歷史數據) [cite: 710]
        history_ret = log_ret.loc[:d].tail(126)[available_tickers].fillna(0)
        sigma = optimizer.estimate_risk(history_ret)

        # 4. 執行優化
        weights_vA_list[d] = optimizer.optimize(alpha_vA, sigma)
        weights_vB_list[d] = optimizer.optimize(alpha_vB, sigma)

        print(f" ✔️  完成調倉日期: {d.date()} | 資產數: {len(available_tickers)}")

    except Exception as e:
        print(f" ❌  日期 {d} 出錯: {e}")
        continue

# --- 核心修改：將月度權重擴展回每日，並向前填充 (ffill) ---
# 這樣計算 PnL 時，月中沒調倉的日子會延用上個月底的權重
opt_weights_vA = pd.DataFrame(weights_vA_list).T.reindex(test_dates).ffill().fillna(0)
opt_weights_vB = pd.DataFrame(weights_vB_list).T.reindex(test_dates).ffill().fillna(0)

print("\n ✨  每月調倉優化運算完成！")

 🚀  優化器啟動 (每月調倉模式) | 處理月份數: 130 | 域名檢查中...
 ✔️  完成調倉日期: 2015-07-31 | 資產數: 864
 ✔️  完成調倉日期: 2015-08-31 | 資產數: 864
 ✔️  完成調倉日期: 2015-09-30 | 資產數: 864
 ✔️  完成調倉日期: 2015-10-30 | 資產數: 864
 ✔️  完成調倉日期: 2015-11-30 | 資產數: 864
 ✔️  完成調倉日期: 2015-12-31 | 資產數: 864
 ✔️  完成調倉日期: 2016-01-29 | 資產數: 864
 ✔️  完成調倉日期: 2016-02-29 | 資產數: 864
 ✔️  完成調倉日期: 2016-03-31 | 資產數: 864
 ✔️  完成調倉日期: 2016-04-29 | 資產數: 864
 ✔️  完成調倉日期: 2016-05-31 | 資產數: 864
 ✔️  完成調倉日期: 2016-06-30 | 資產數: 864
 ✔️  完成調倉日期: 2016-07-29 | 資產數: 864
 ✔️  完成調倉日期: 2016-08-31 | 資產數: 864
 ✔️  完成調倉日期: 2016-09-30 | 資產數: 864
 ✔️  完成調倉日期: 2016-10-31 | 資產數: 864
 ✔️  完成調倉日期: 2016-11-30 | 資產數: 864
 ✔️  完成調倉日期: 2016-12-30 | 資產數: 864
 ✔️  完成調倉日期: 2017-01-31 | 資產數: 864
 ✔️  完成調倉日期: 2017-02-28 | 資產數: 864
 ✔️  完成調倉日期: 2017-03-31 | 資產數: 864
 ✔️  完成調倉日期: 2017-04-28 | 資產數: 864
 ✔️  完成調倉日期: 2017-05-31 | 資產數: 864
 ✔️  完成調倉日期: 2017-06-30 | 資產數: 864
 ✔️  完成調倉日期: 2017-07-31 | 資產數: 864
 ✔️  完成調倉日期: 2017-08-31 | 資產數: 864
 ✔️  完成調倉日期: 2017-09-29 | 資產數: 864
 ✔️  完成調倉日期:

In [36]:
# @title Performance & Attribution Report
from sklearn.decomposition import PCA
import statsmodels.api as sm
import pandas as pd
import numpy as np

def run_comprehensive_eval(opt_weights, strategy_name):
    print(f"\n{'='*30}\n 📊 正在生成 {strategy_name} 完整分析報告\n{'='*30}")

    # 1. 基礎數據準備
    s_ret = simple_ret.loc[opt_weights.index]
    # 策略收益：昨日權重 * 今日回報 [cite: 170]
    strat_ret = (opt_weights.shift(1) * s_ret).sum(axis=1).fillna(0)

    # 統計用數據
    l_ret = log_ret.loc[opt_weights.index].fillna(0)
    strat_log_ret = np.log(1 + strat_ret).replace([np.inf, -np.inf], 0).fillna(0)

    years = sorted(opt_weights.index.year.unique())
    perf_data = []
    attr_data = []

    for yr in years:
        yr_mask = opt_weights.index.year == yr
        y_strat_ret = strat_ret[yr_mask]
        y_strat_log_ret = strat_log_ret[yr_mask]
        y_asset_log_ret = l_ret[yr_mask]
        y_weights = opt_weights[yr_mask]

        if len(y_strat_ret) < 20: continue

        # --- A. 績效報告指標 (Performance Report) ---
        # 1. Returns & Sharpe [cite: 248-249]
        ann_ret = (1 + y_strat_ret).prod() - 1
        sharpe = (y_strat_ret.mean() / (y_strat_ret.std() + 1e-9)) * np.sqrt(252)

        # 2. Rank IC & IR (使用次日對數回報) [cite: 257, 260]
        y_next_ret = log_ret.shift(-1).loc[y_weights.index].fillna(0)
        daily_rank_ic = y_weights.corrwith(y_next_ret, axis=1, method='spearman').fillna(0)
        rank_ic_mean = daily_rank_ic.mean()
        ir = (rank_ic_mean / (daily_rank_ic.std() + 1e-9)) * np.sqrt(252)

        # 3. Turnover, Drawdown & Margin [cite: 252, 265-266]
        y_diff = y_weights.diff().abs().sum(axis=1)
        y_gross = y_weights.abs().sum(axis=1).replace(0, 1)
        turnover = (y_diff / y_gross).mean()

        cum_pnl = (1 + y_strat_ret).cumprod()
        drawdown = abs(((cum_pnl / cum_pnl.cummax()) - 1).min())

        margin_bp = (y_strat_ret.sum() / (y_diff.sum() + 1e-9) * 10000)

        # 4. Fitness (WQ Formula)
        fitness_denom = max(turnover, 0.125)
        fitness = sharpe * np.sqrt(abs(ann_ret) / fitness_denom)

        perf_data.append({
            "Year": yr,
            "Rank IC": round(rank_ic_mean, 4),
            "IR": round(ir, 2),
            "Sharpe": round(sharpe, 2),
            "Fitness": round(fitness, 2),
            "Returns": f"{ann_ret:.2%}",
            "Turnover": f"{turnover:.2%}",
            "Drawdown": f"{drawdown:.2%}",
            "Margin": f"{margin_bp:.2f} bp"
        })

        # --- B. 歸因報告指標 (Attribution Report) ---
        # 利用 PCA 提取前 5 大主成分
        pca_model = PCA(n_components=5)
        risk_factors = pca_model.fit_transform(y_asset_log_ret)

        X = sm.add_constant(risk_factors)
        reg = sm.OLS(y_strat_log_ret, X).fit()

        sys_ratio = reg.rsquared # 系統性比例
        total_log = y_strat_log_ret.sum()

        attr_data.append({
            "Year": yr,
            "Total Log Ret": f"{total_log:.2%}",
            "Systematic (PCA)": f"{sys_ratio:.1%}",
            "Specific (Alpha)": f"{(1 - sys_ratio):.1%}",
            "Alpha Contribution": f"{(total_log * (1 - sys_ratio)):.2%}"
        })

    # 轉為 DataFrame
    df_perf = pd.DataFrame(perf_data).set_index("Year")
    df_attr = pd.DataFrame(attr_data).set_index("Year")

    return df_perf, df_attr

# 執行分析
# vA: EW opt (等權合成 + 優化)
# vB: MW opt (MW 動態合成 + 優化)
perf_ew, attr_ew = run_comprehensive_eval(opt_weights_vA, "EW opt")
perf_mw, attr_mw = run_comprehensive_eval(opt_weights_vB, "MW opt")

# 顯示 4 張表格
print("\n[ 1. EW optimized - Performance Report ]")
display(perf_ew)
print("\n[ 2. EW optimized - Attribution Report ]")
display(attr_ew)
print("\n[ 3. MW optimized - Performance Report ]")
display(perf_mw)
print("\n[ 4. MW optimized - Attribution Report ]")
display(attr_mw)


 📊 正在生成 EW opt 完整分析報告


/usr/local/lib/python3.12/dist-packages/pandas/core/nanops.py:1632: ConstantInputWarning:

An input array is constant; the correlation coefficient is not defined.

/usr/local/lib/python3.12/dist-packages/pandas/core/nanops.py:1632: ConstantInputWarning:

An input array is constant; the correlation coefficient is not defined.

/usr/local/lib/python3.12/dist-packages/pandas/core/nanops.py:1632: ConstantInputWarning:

An input array is constant; the correlation coefficient is not defined.

/usr/local/lib/python3.12/dist-packages/pandas/core/nanops.py:1632: ConstantInputWarning:

An input array is constant; the correlation coefficient is not defined.

/usr/local/lib/python3.12/dist-packages/pandas/core/nanops.py:1632: ConstantInputWarning:

An input array is constant; the correlation coefficient is not defined.

/usr/local/lib/python3.12/dist-packages/pandas/core/nanops.py:1632: ConstantInputWarning:

An input array is constant; the correlation coefficient is not defined.

/usr/local/lib/p


 📊 正在生成 MW opt 完整分析報告


/usr/local/lib/python3.12/dist-packages/pandas/core/nanops.py:1632: ConstantInputWarning:

An input array is constant; the correlation coefficient is not defined.

/usr/local/lib/python3.12/dist-packages/pandas/core/nanops.py:1632: ConstantInputWarning:

An input array is constant; the correlation coefficient is not defined.

/usr/local/lib/python3.12/dist-packages/pandas/core/nanops.py:1632: ConstantInputWarning:

An input array is constant; the correlation coefficient is not defined.

/usr/local/lib/python3.12/dist-packages/pandas/core/nanops.py:1632: ConstantInputWarning:

An input array is constant; the correlation coefficient is not defined.

/usr/local/lib/python3.12/dist-packages/pandas/core/nanops.py:1632: ConstantInputWarning:

An input array is constant; the correlation coefficient is not defined.

/usr/local/lib/python3.12/dist-packages/pandas/core/nanops.py:1632: ConstantInputWarning:

An input array is constant; the correlation coefficient is not defined.

/usr/local/lib/p


[ 1. EW optimized - Performance Report ]


...

,Total Log Ret,Systematic (PCA),Specific (Alpha),Alpha Contribution
Year,,,,
2015,1.35%,3.4%,96.6%,1.30%
2016,12.46%,13.7%,86.3%,10.75%
2017,1.04%,6.3%,93.7%,0.98%
2018,2.06%,5.3%,94.7%,1.95%
2019,0.01%,30.4%,69.6%,0.00%
2020,12.81%,10.8%,89.2%,11.42%
2021,8.35%,4.1%,95.9%,8.00%
2022,6.64%,8.9%,91.1%,6.05%
2023,-6.59%,4.5%,95.5%,-6.29%



[ 3. MW optimized - Performance Report ]


,Rank IC,IR,Sharpe,Fitness,Returns,Turnover,Drawdown,Margin
Year,,,,,,,,
2015,-0.0197,-1.86,0.17,0.03,0.52%,8.57%,3.97%,6.39 bp
2016,-0.0131,-1.09,1.39,1.58,16.01%,8.93%,6.86%,68.73 bp
2017,0.0039,0.49,-0.57,-0.33,-4.09%,8.57%,10.62%,-18.33 bp
2018,0.0110,1.07,-0.14,-0.04,-1.20%,8.13%,9.44%,-4.70 bp
2019,0.0021,0.15,0.21,0.07,1.41%,8.65%,9.55%,8.06 bp
2020,0.0282,1.97,0.63,0.50,7.87%,7.27%,16.43%,45.93 bp
2021,0.0140,1.53,1.43,1.26,9.69%,4.80%,4.82%,78.24 bp
2022,-0.0013,-0.09,0.73,0.55,7.12%,7.65%,7.43%,38.53 bp
2023,-0.0021,-0.19,-0.93,-0.67,-6.50%,7.64%,9.64%,-33.90 bp



[ 4. MW optimized - Attribution Report ]


,Total Log Ret,Systematic (PCA),Specific (Alpha),Alpha Contribution
Year,,,,
2015,0.52%,4.7%,95.3%,0.50%
2016,14.85%,11.7%,88.3%,13.11%
2017,-4.18%,4.7%,95.3%,-3.98%
2018,-1.21%,6.0%,94.0%,-1.14%
2019,1.40%,28.8%,71.2%,1.00%
2020,7.57%,25.9%,74.1%,5.61%
2021,9.25%,4.4%,95.6%,8.84%
2022,6.88%,11.8%,88.2%,6.07%
2023,-6.72%,4.1%,95.9%,-6.45%


In [41]:
# @title Alpha Statistical Significance Test
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.decomposition import PCA

def test_alpha_significance(opt_weights, strategy_name):
    print(f"\n{'='*45}\n 🔬 執行 {strategy_name} 的 Alpha 統計顯著性檢定\n{'='*45}")

    # 1. 準備數據
    s_ret = simple_ret.loc[opt_weights.index]
    strat_ret = (opt_weights.shift(1) * s_ret).sum(axis=1).fillna(0)
    strat_log_ret = np.log(1 + strat_ret).replace([np.inf, -np.inf], 0).fillna(0)
    y_asset_log_ret = log_ret.loc[opt_weights.index].fillna(0)

    # 2. 提取整體時間區段的共同風險因子 (PCA Top 5)
    pca_model = PCA(n_components=5)
    risk_factors = pca_model.fit_transform(y_asset_log_ret)

    # 3. 執行全樣本線性回歸
    X = sm.add_constant(risk_factors)
    model = sm.OLS(strat_log_ret, X).fit()

    # 【核心修復】：特質報酬序列 u_P = 常數項 (Alpha期望值) + 殘差 (Alpha波動)
    # model.params[0] 是 sm.add_constant 加上的常數項 intercept
    u_P = model.params[0] + model.resid

    # 4. 計算 Information Ratio (IR) 與 t-stat
    daily_mean_alpha = u_P.mean()
    daily_std_alpha = u_P.std()

    # 計算年化 IR
    ir_annual = (daily_mean_alpha / (daily_std_alpha + 1e-9)) * np.sqrt(252)

    # 計算總年數
    years = len(u_P) / 252

    # 計算 t-stat (公式：IR * sqrt(Years))
    t_stat = ir_annual * np.sqrt(years)

    # 5. 判斷是否具備統計顯著性 (95% 信心水準 t > 2)
    is_significant = t_stat > 2

    # 輸出檢定結果報告
    print(f"▶ 測試期間總交易日: {len(u_P)} 天 (約 {years:.2f} 年)")
    print(f"▶ Alpha 年化 IR: {ir_annual:.4f}")
    print(f"▶ 檢定 t-統計量 (t-stat): {t_stat:.4f}")
    print("-" * 45)

    if is_significant:
        print("✅ 結論：t-stat > 2")
        print("Alpha 具備 95% 統計顯著性")
    elif t_stat > 0:
        print("⚠️ 結論：0 < t-stat <= 2")
        print("Alpha 為正，但尚未達到 95% 統計顯著性。")
    else:
        print("❌ 結論：t-stat < 0")
        print("Alpha 為負，代表策略在剔除共同因素後，並未能產生正向 Alpha。")

# 針對兩種合成優化結果分別執行檢定
test_alpha_significance(opt_weights_vA, "EW opt (等權合成)")
test_alpha_significance(opt_weights_vB, "MW opt (動態權重合成)")


 🔬 執行 EW opt (等權合成) 的 Alpha 統計顯著性檢定


/tmp/ipykernel_6495/316572351.py:26: FutureWarning:

Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`



▶ 測試期間總交易日: 2706 天 (約 10.74 年)
▶ Alpha 年化 IR: 0.2682
▶ 檢定 t-統計量 (t-stat): 0.8790
---------------------------------------------
⚠️ 結論：0 < t-stat <= 2
Alpha 為正，但尚未達到 95% 統計顯著性。

 🔬 執行 MW opt (動態權重合成) 的 Alpha 統計顯著性檢定
▶ 測試期間總交易日: 2706 天 (約 10.74 年)
▶ Alpha 年化 IR: 0.1860
▶ 檢定 t-統計量 (t-stat): 0.6095
---------------------------------------------
⚠️ 結論：0 < t-stat <= 2
Alpha 為正，但尚未達到 95% 統計顯著性。


/tmp/ipykernel_6495/316572351.py:26: FutureWarning:

Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`

